In [119]:
nvars = 4 # number of variables
BR = QQ[",".join("x"+str(i) for i in range(1, nvars+1))+",z"] # polynomial ring in x1, ..., xnvars, z
# BR is a global variable
BR.inject_variables() # make it so you can use those variables
SymmetricFunctions(QQ).inject_shorthands(verbose=False) # define the bases of symmetric functions

def do_P_k(k, n):
    # we are computing p_k[s_n] = s_n[p_k]
    # evaluate at the generators gens = BR.gens()[:-1] = (x1, x2, x3, x4)
    global BR,nvars
    CR = s[[]].expand(nvars).parent()
    return s[n].expand(nvars).subs({CR('x'+str(i)) : BR('x'+str(i+1))**k for i in range(nvars)})
@cached_function
def do_P_lambda(la, n):
    global BR,nvars
    a=expand(mul(do_P_k(p, n) for p in la)*mul(BR('x'+str(i+1))-BR('x'+str(j+1)) for i in range(nvars) for j in range(i+1, nvars)))
    return sum(c*mul(BR('x'+str(i+1))^(v[i]-(nvars-i-1)) for i in range(nvars))\
               for (v,c) in a.monomial_coefficients().items() if all(v[i]>v[i+1] for i in range(nvars-1)))
@cached_function
def den_coeff(d):
    global BR
    gz = BR.gens()[-1] # the last variable is the z
    return BR(SR(den_guess()).coefficient(gz,d))
def calc_num(la, d):
    return sum(den_coeff(d-r)*do_P_lambda(Partition(la),r) for r in range(d+1))

"""
This is the denominator that you are trying to determine
"""
@cached_function
def den_guess():
    global BR
    #m1 = x1*x2*x3*z
    #m2 = x1^3*z
    #m3 = x1^3*x2^3*z^2
    return BR((1 - z*x1**4) * (1 - z*x1**3*x2) * (1 - z*x1**2*x2**2) * (1 - z**3*x1**6*x2**6) * (1 - z*x1**2*x2*x3) * (1 - z**3*x1**6*x2**3*x3**3) * (1 - z**2*x1**3*x2**3*x3**2) * (1 - z**3*x1**4*x2**4*x3**4) * (1 - z*x1*x2*x3*x4)) # My default start point is den_guess is 1, but you build it up from there.

Defining x1, x2, x3, x4, z


In [120]:
'''
#Version that works with older versions of Sage
@cached_function
def do_P_lambda(la, n):
    global BR, nvars
    a = expand(mul(do_P_k(p, n) for p in la) * mul(BR('x'+str(i+1)) - BR('x'+str(j+1)) for i in range(nvars) for j in range(i+1, nvars)))
    
    # Change is on the line below: swap .monomial_coefficients() for .dict()
    return sum(c * mul(BR('x'+str(i+1))^(v[i] - (nvars-i-1)) for i in range(nvars)) \
               for (v,c) in a.dict().items() if all(v[i] > v[i+1] for i in range(nvars-1)))
'''

"\n#Version that works with older versions of Sage\n@cached_function\ndef do_P_lambda(la, n):\n    global BR, nvars\n    a = expand(mul(do_P_k(p, n) for p in la) * mul(BR('x'+str(i+1)) - BR('x'+str(j+1)) for i in range(nvars) for j in range(i+1, nvars)))\n\n    # Change is on the line below: swap .monomial_coefficients() for .dict()\n    return sum(c * mul(BR('x'+str(i+1))^(v[i] - (nvars-i-1)) for i in range(nvars))                for (v,c) in a.dict().items() if all(v[i] > v[i+1] for i in range(nvars-1)))\n"

In [124]:
out=0
for d in range(0,25):
    CC = calc_num([3,1],d)
    print(d,len(list(CC)), list(CC)[:3])
    # d is the degree
    # len(list(CC)) is how "big" the expression is at degree d
    # list(CC)[:3] gives three terms of the set of all terms in the expression
    #   (you can try replacing this with list(CC)[-3:] if you don't see patterns)
    print("*************")
    out+=z**d*CC

0 1 [(1, 1)]
*************
1 3 [(-1, x1^3*x2), (-2, x1^2*x2^2), (-1, x1^2*x2*x3)]
*************
2 5 [(1, x1^6*x2^2), (2, x1^5*x2^3), (1, x1^4*x2^4)]
*************
3 7 [(-2, x1^8*x2^4), (-1, x1^7*x2^5), (-1, x1^8*x2^3*x3)]
*************
4 12 [(1, x1^10*x2^6), (2, x1^10*x2^5*x3), (2, x1^9*x2^6*x3)]
*************
5 13 [(-1, x1^12*x2^7*x3), (-1, x1^11*x2^8*x3), (1, x1^11*x2^7*x3^2)]
*************
6 13 [(2, x1^14*x2^7*x3^3), (-1, x1^13*x2^8*x3^3), (-1, x1^12*x2^9*x3^3)]
*************
7 13 [(-1, x1^16*x2^9*x3^3), (1, x1^15*x2^10*x3^3), (1, x1^14*x2^11*x3^3)]
*************
8 12 [(1, x1^18*x2^10*x3^4), (1, x1^17*x2^11*x3^4), (-1, x1^16*x2^12*x3^4)]
*************
9 7 [(2, x1^19*x2^11*x3^6), (1, x1^17*x2^13*x3^6), (-3, x1^18*x2^11*x3^7)]
*************
10 5 [(-1, x1^21*x2^13*x3^6), (3, x1^20*x2^13*x3^7), (1, x1^20*x2^12*x3^8)]
*************
11 3 [(-1, x1^22*x2^15*x3^7), (-2, x1^22*x2^14*x3^8), (-1, x1^21*x2^15*x3^8)]
*************
12 1 [(1, x1^24*x2^16*x3^8)]
*************
13 0 []
*************
1

In [122]:
factor(out)

(x1^2*x2*x3*z - 1)^2 * (x1^2*x2^2*z - 1)^2 * (x1^16*x2^10*x3^6*z^8 + x1^14*x2^9*x3^5*z^7 - x1^13*x2^9*x3^6*z^7 - x1^13*x2^7*x3^4*z^6 + x1^12*x2^8*x3^4*z^6 + x1^12*x2^7*x3^5*z^6 - 2*x1^11*x2^8*x3^5*z^6 + x1^10*x2^8*x3^6*z^6 - 2*x1^11*x2^6*x3^3*z^5 + x1^10*x2^7*x3^3*z^5 + 2*x1^10*x2^6*x3^4*z^5 - 2*x1^9*x2^7*x3^4*z^5 - x1^9*x2^6*x3^5*z^5 + x1^8*x2^7*x3^5*z^5 + x1^10*x2^4*x3^2*z^4 - 2*x1^9*x2^5*x3^2*z^4 - x1^9*x2^4*x3^3*z^4 + 4*x1^8*x2^5*x3^3*z^4 - x1^7*x2^6*x3^3*z^4 - 2*x1^7*x2^5*x3^4*z^4 + x1^6*x2^6*x3^4*z^4 + x1^8*x2^3*x3*z^3 - x1^7*x2^4*x3*z^3 - 2*x1^7*x2^3*x3^2*z^3 + 2*x1^6*x2^4*x3^2*z^3 + x1^6*x2^3*x3^3*z^3 - 2*x1^5*x2^4*x3^3*z^3 + x1^6*x2^2*z^2 - 2*x1^5*x2^2*x3*z^2 + x1^4*x2^3*x3*z^2 + x1^4*x2^2*x3^2*z^2 - x1^3*x2^3*x3^2*z^2 - x1^3*x2*z + x1^2*x2*x3*z + 1)

In [83]:
out

x1^4*x2^2*z^2 + x1^2*x2*z + 1

In [56]:
factor(den_guess())

(-1) * (x1*x2*x3*z + 1) * (x1^2*x2*z - 1) * (x1^3*z - 1) * (x1^3*x2^3*z^2 - 1) * (x1^3*x2^3*z^2 + 1)

In [123]:
out/den_guess()

(x1^16*x2^10*x3^6*z^8 + x1^14*x2^9*x3^5*z^7 - x1^13*x2^9*x3^6*z^7 - x1^13*x2^7*x3^4*z^6 + x1^12*x2^8*x3^4*z^6 + x1^12*x2^7*x3^5*z^6 - 2*x1^11*x2^8*x3^5*z^6 + x1^10*x2^8*x3^6*z^6 - 2*x1^11*x2^6*x3^3*z^5 + x1^10*x2^7*x3^3*z^5 + 2*x1^10*x2^6*x3^4*z^5 - 2*x1^9*x2^7*x3^4*z^5 - x1^9*x2^6*x3^5*z^5 + x1^8*x2^7*x3^5*z^5 + x1^10*x2^4*x3^2*z^4 - 2*x1^9*x2^5*x3^2*z^4 - x1^9*x2^4*x3^3*z^4 + 4*x1^8*x2^5*x3^3*z^4 - x1^7*x2^6*x3^3*z^4 - 2*x1^7*x2^5*x3^4*z^4 + x1^6*x2^6*x3^4*z^4 + x1^8*x2^3*x3*z^3 - x1^7*x2^4*x3*z^3 - 2*x1^7*x2^3*x3^2*z^3 + 2*x1^6*x2^4*x3^2*z^3 + x1^6*x2^3*x3^3*z^3 - 2*x1^5*x2^4*x3^3*z^3 + x1^6*x2^2*z^2 - 2*x1^5*x2^2*x3*z^2 + x1^4*x2^3*x3*z^2 + x1^4*x2^2*x3^2*z^2 - x1^3*x2^3*x3^2*z^2 - x1^3*x2*z + x1^2*x2*x3*z + 1)/(-x1^23*x2^15*x3^9*x4*z^12 + x1^22*x2^14*x3^8*z^11 - x1^21*x2^14*x3^8*x4*z^11 - x1^21*x2^13*x3^9*x4*z^11 + x1^20*x2^14*x3^9*x4*z^11 + x1^19*x2^15*x3^9*x4*z^11 + x1^20*x2^13*x3^7*z^10 + x1^20*x2^12*x3^8*z^10 - x1^19*x2^13*x3^8*z^10 - x1^18*x2^14*x3^8*z^10 + x1^20*x2^12*x3^7*x

In [25]:
nvars = 1
target_partition = [1]

BR = QQ[",".join("x"+str(i) for i in range(1, nvars+1))+",z"]
BR.inject_variables()
SymmetricFunctions(QQ).inject_shorthands(verbose=False)

def do_P_k(k, n):
    global BR, nvars
    CR = s[1].expand(nvars).parent()
    return s[n].expand(nvars).subs({CR.gens()[i] : BR('x'+str(i+1))**k for i in range(nvars)})

@cached_function
def do_P_lambda(la, n):
    global BR, nvars
    a = BR(expand(mul(do_P_k(p, n) for p in la) * mul(BR('x'+str(i+1)) - BR('x'+str(j+1)) for i in range(nvars) for j in range(i+1, nvars))))
    return sum(c * mul(BR('x'+str(i+1))^(v[i] - (nvars-i-1)) for i in range(nvars)) \
               for (v,c) in a.dict().items() if all(v[i] > v[i+1] for i in range(nvars-1)))

@cached_function
def den_coeff(d):
    global BR
    gz = BR.gens()[-1]
    return BR(SR(den_guess()).coefficient(gz, d))

def calc_num(la, d):
    return sum(den_coeff(d-r) * do_P_lambda(Partition(la), r) for r in range(d+1))

@cached_function
def den_guess():
    global BR
    return BR(1 - z*x1)

out = 0
for d in range(0, 20):
    CC = calc_num(target_partition, d)
    if CC:
        print(d, len(list(CC)), list(CC)[:3])
    else:
        print(d, 0, "[]")
    
    out += z**d * CC

Defining x1, z
0 1 [(1, 1)]
1 0 []
2 0 []
3 0 []
4 0 []
5 0 []
6 0 []
7 0 []
8 0 []
9 0 []
10 0 []
11 0 []
12 0 []
13 0 []
14 0 []
15 0 []
16 0 []
17 0 []
18 0 []
19 0 []


In [27]:
out/den_guess()

1/(-x1*z + 1)

The following code can be used to compute
## $${\tilde P}_\mu(x_1, x_2, \ldots, x_d;z) = \sum_{n \geq 0} p_\mu[s_n](x_1,x_2,\ldots,x_d)  z^n$$

There are two problems with this approach.

1. It is incredibly slow because it relies on the Sage implementation of the `MacMahonOmega` operator.
2. Once you compute ${\tilde P}_\mu(x_1, x_2, \ldots, x_d;z)$, you still need to apply ${\mathcal L}_{X_d}$ to the expression for it to be useful for what we need.

In [18]:
Sym = SymmetricFunctions(QQ)
Sym.inject_shorthands(verbose=False)
R = PolynomialRing(QQ, 'a, b, x1, x2, x3, x4, z').fraction_field()
R.inject_variables()
x = R.gens()[2:-1]
a = R.gens()[0]
b = R.gens()[1]
z = R.gens()[-1]
def normalize_rational_function(Q):
    # Normalize denominators
    S = PolynomialRing(PolynomialRing(QQ, 'a,b'), 'x1,x2,x3,x4,z')
    K = R.fraction_field()
    den = []
    factors = Q.denominator().factor()
    scalar = factors.unit()
    for (factor, exp) in factors:
        c = S(factor).constant_coefficient()
        den.append((K(factor) / c, exp))
        scalar *= c**exp
    den = Factorization(den)
    num = Q.numerator() / scalar
    return (num, den)
def latex_fraction(X):
    if X == 0:
        return LatexExpr("0")
    num, den = normalize_rational_function(X)
    return LatexExpr(f"\\frac{{{latex(num.factor())}}}{{{latex(den)}}}")
def CT(f, g):
    # given two polynomials f(z) and g(z), compute the constant term of f(z/a) * g(a)
    Q = f.subs({z: z / (a * b)}) * g.subs({z: a * b})
    num, den = normalize_rational_function(Q)
    PTa = MacMahonOmega(a, num, den)
    CTa = prod(PTa).subs(b=0)
    return CTa
def P(k):
    return R.one() / R.prod((1 - z * xi**k) for xi in x)

Defining a, b, x1, x2, x3, x4, z


The programs above can be used to compute the following expressions *in theory* and *given enough computer time*, but I haven't found that they work in practice.

In [2]:
P11 = CT(P(1), P(1))

In [3]:
P11

(-x1^3*x2^3*x3^3*x4^3*z^6 - x1^3*x2^3*x3^2*x4^2*z^5 - x1^3*x2^2*x3^3*x4^2*z^5 - x1^2*x2^3*x3^3*x4^2*z^5 - x1^3*x2^2*x3^2*x4^3*z^5 - x1^2*x2^3*x3^2*x4^3*z^5 - x1^2*x2^2*x3^3*x4^3*z^5 + x1^3*x2^2*x3^2*x4*z^4 + x1^2*x2^3*x3^2*x4*z^4 + x1^2*x2^2*x3^3*x4*z^4 + x1^3*x2^2*x3*x4^2*z^4 + x1^2*x2^3*x3*x4^2*z^4 + x1^3*x2*x3^2*x4^2*z^4 + 3*x1^2*x2^2*x3^2*x4^2*z^4 + x1*x2^3*x3^2*x4^2*z^4 + x1^2*x2*x3^3*x4^2*z^4 + x1*x2^2*x3^3*x4^2*z^4 + x1^2*x2^2*x3*x4^3*z^4 + x1^2*x2*x3^2*x4^3*z^4 + x1*x2^2*x3^2*x4^3*z^4 - x1^2*x2^2*x3^2*z^3 + x1^3*x2*x3*x4*z^3 + x1*x2^3*x3*x4*z^3 + x1*x2*x3^3*x4*z^3 - x1^2*x2^2*x4^2*z^3 - x1^2*x3^2*x4^2*z^3 - x2^2*x3^2*x4^2*z^3 + x1*x2*x3*x4^3*z^3 - x1^2*x2*x3*z^2 - x1*x2^2*x3*z^2 - x1*x2*x3^2*z^2 - x1^2*x2*x4*z^2 - x1*x2^2*x4*z^2 - x1^2*x3*x4*z^2 - 3*x1*x2*x3*x4*z^2 - x2^2*x3*x4*z^2 - x1*x3^2*x4*z^2 - x2*x3^2*x4*z^2 - x1*x2*x4^2*z^2 - x1*x3*x4^2*z^2 - x2*x3*x4^2*z^2 + x1*x2*z + x1*x3*z + x2*x3*z + x1*x4*z + x2*x4*z + x3*x4*z + 1)/(x1^5*x2^5*x3^5*x4^5*z^10 - x1^5*x2^5*x3^5*x4^3*z

In [ ]:
P111 = CT(P(1), P11)

In [ ]:
P1111 = CT(P111, P(1))

In [ ]:
P211 = CT(P(2), P11)

In [ ]:
P22 = CT(P(2), P(2))

In [ ]:
P31 = CT(P(3), P(1))

In [ ]:
P4 = P(4)